# Unipolar PAM-4 IM/DD — End-to-End Autoencoder

Eye diagram and BER vs $E_b/N_0$ for the 4-segment-MZM + dispersive-fiber link
(DPD at the transmitter, FFE at the receiver, trained jointly).

**Just run all cells.** Pick the noise regime with `REGIME` below:
- `"ase"` — optical ASE beat noise on the field (thermal/electrical noise off), theory = Forestieri A.47
- `"thermal"` — electrical/electronics (thermal) noise at the ADC (optical noise off), theory = Forestieri A.19

Each regime has its own checkpoint `results/pam4_<regime>.pt`: if it exists it is loaded,
otherwise the autoencoder is trained from scratch and saved. Set `FORCE_RETRAIN = True`
(or delete the checkpoint) to retrain, e.g. after editing `config.py`.

Run this notebook from the `MLforCPO2` folder so the local modules and `results/` path resolve.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
sys.path.insert(0, "functions")        # local library modules live in functions/

# auto-reload edited .py modules without restarting the kernel
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.signal import resample_poly

from config import Config
from channel import OpticalChannel
from transmitter import Transmitter, bits_to_symbols
from receiver import Receiver
from train import train, evaluate, random_bits
from utils import theoretical_ber_unipolar, theoretical_ber_unipolar_ase, decision_phase_by_separation

%matplotlib inline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ----- knobs -----
REGIME = "ase"           # "ase": optical ASE beat noise (thermal off) | "thermal": electrical noise (optical off)
NUM_STEPS = 10000        # training steps when no checkpoint is found
NUM_EVAL = 400000        # symbols per Eb/N0 point in the waterfall
FORCE_RETRAIN = False    # True -> ignore an existing checkpoint and retrain

config = Config()
config.noise_regime = REGIME
config.minibatch_symbols = 4096
config.edge_guard_symbols = 48

# one checkpoint per regime, so switching REGIME never loads a mismatched model
CHECKPOINT = os.path.join("results", f"pam4_{REGIME}.pt")
print(config.summary())

## Load the trained autoencoder (or train it if missing)

In [ ]:
def load_or_train(config, device, num_steps, checkpoint_path, num_eval, force_retrain=False):
    """Return tx, channel, rx, ebn0_db, measured — loading a checkpoint if present.
    On load, the build-time choices that are NOT in the state_dict (sqrt-companding of the
    receiver, optical-filter type of the channel) are restored from the checkpoint, so the
    rebuilt models always match the trained weights."""
    transmitter = Transmitter(config).to(device)

    if os.path.exists(checkpoint_path) and not force_retrain:
        print(f"loading checkpoint: {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, weights_only=False, map_location=device)
        config.rx_sqrt_companding = ckpt.get("rx_sqrt_companding", config.rx_sqrt_companding)
        config.optical_filter_type = ckpt.get("optical_filter_type", config.optical_filter_type)
        channel = OpticalChannel(config).to(device)
        receiver = Receiver(config).to(device)
        transmitter.load_state_dict(ckpt["transmitter"])
        receiver.load_state_dict(ckpt["receiver"])
        ebn0_db = np.asarray(ckpt["ebn0_db"])
        measured = np.asarray(ckpt["measured"])
    else:
        print("no checkpoint (or FORCE_RETRAIN) -> training from scratch")
        torch.manual_seed(0)
        np.random.seed(0)
        transmitter, channel, receiver = train(config, device, num_steps=num_steps)
        ebn0_db = np.arange(6, 22, 2.0)
        measured = evaluate(transmitter, channel, receiver, config, ebn0_db, num_eval, device)
        os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
        torch.save({
            "transmitter": transmitter.state_dict(),
            "receiver": receiver.state_dict(),
            "config": {k: v for k, v in vars(config).items()},
            "rx_sqrt_companding": bool(receiver.sqrt_companding),
            "optical_filter_type": channel.optical_filter_type,
            "ebn0_db": ebn0_db,
            "measured": measured,
        }, checkpoint_path)
        print(f"saved checkpoint: {checkpoint_path}")

    transmitter.eval()
    receiver.eval()
    return transmitter, channel, receiver, ebn0_db, measured


transmitter, channel, receiver, ebn0_db, measured = load_or_train(
    config, device, NUM_STEPS, CHECKPOINT, NUM_EVAL, FORCE_RETRAIN)

## Eye diagram (photodetected, noiseless)

In [ ]:
def draw_eye(ax, photocurrent, samples_per_symbol, phase, title,
             fine_sps=40, skip=100, num_symbols=4000):
    fine = resample_poly(photocurrent[:(skip + num_symbols + 8) * samples_per_symbol],
                         fine_sps, samples_per_symbol)
    offset = int(round(phase * fine_sps / samples_per_symbol))
    centers = (skip + np.arange(num_symbols)) * fine_sps + offset
    keep = (centers - fine_sps >= 0) & (centers + fine_sps < len(fine))
    centers = centers[keep]
    window = np.arange(-fine_sps, fine_sps)[:, None] + centers[None, :]
    ax.plot(np.arange(-fine_sps, fine_sps) / fine_sps, fine[window],
            color=(0.85, 0.33, 0.10, 0.03), linewidth=0.4)
    ax.axvline(0, color="k", linestyle=":", linewidth=1)
    ax.set_xlim(-1, 1)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Symbol time")
    ax.set_ylabel("Photodetected power")
    ax.set_title(title)


sps = config.samples_per_symbol_sim
with torch.no_grad():
    bits = random_bits(config.bits_per_symbol, 6000, device)
    symbols = bits_to_symbols(bits, config.bits_per_symbol).cpu().numpy()
    photocurrent = channel(transmitter(bits)).cpu().numpy()   # noiseless levels

phase, levels = decision_phase_by_separation(photocurrent, sps, symbols)
gaps = np.diff(np.sort(levels))
print(f"level means (sym 0..3): {np.round(levels, 4)}")
print(f"sorted gaps: {np.round(gaps, 4)}   equispacing std/mean = {gaps.std() / gaps.mean():.3f}")
# In the ASE regime the optimum is equispaced AMPLITUDE -> intensity ~ [0,1,4,9],
# so the intensity gaps are deliberately UNequal (~1:3:5), not equispaced.

fig_eye, ax_eye = plt.subplots(figsize=(7, 5.5))
draw_eye(ax_eye, photocurrent, sps, phase, f"Unipolar PAM-4 eye ({config.noise_regime} regime)")
fig_eye.tight_layout()
fig_eye.savefig(os.path.join("results", "pam4_eye.png"), dpi=150, bbox_inches="tight")
plt.show()

## BER vs $E_b/N_0$ curves

In [ ]:
if config.noise_regime == "ase":
    theory = theoretical_ber_unipolar_ase(ebn0_db, config.modulation_order)
    theory_label = "Theory unipolar PAM-4 (ASE, A.47)"
else:
    theory = theoretical_ber_unipolar(ebn0_db, config.modulation_order)
    theory_label = "Theory unipolar PAM-4 (thermal, A.19)"
measured_floor = np.maximum(measured, 1.0 / NUM_EVAL)   # smallest resolvable BER

print("  Eb/N0     measured     theory")
for e, m, t in zip(ebn0_db, measured, theory):
    print(f"  {e:5.1f}    {m:.3e}    {t:.3e}")

fig_ber, ax_ber = plt.subplots(figsize=(7, 5.5))
ax_ber.semilogy(ebn0_db, measured_floor, "-o", linewidth=2, label="E2E autoencoder (DPD + FFE)")
ax_ber.semilogy(ebn0_db, theory, "k--", linewidth=1.5, label=theory_label)
ax_ber.grid(True, which="both", alpha=0.3)
ax_ber.set_xlabel(r"$E_b/N_0$ [dB]")
ax_ber.set_ylabel("BER")
ax_ber.set_ylim(1e-9, 1)
ax_ber.set_title(f"BER vs $E_b/N_0$  ({config.noise_regime} noise)")
ax_ber.legend(loc="lower left")
fig_ber.tight_layout()
fig_ber.savefig(os.path.join("results", "pam4_ber.png"), dpi=150, bbox_inches="tight")
plt.show()